# Additional file 1 — SUPPORT2 analyses

**Valid inference with coarsened and missing covariates in critical care prognostic models: a hybrid probability–possibility framework**

This notebook reproduces every SUPPORT2 result in the paper:

| Section | Paper |
|---|---|
| 4. Main simulation, M1–M4, 1,000 replicates (HPPSM by vertices + axes, adaptive search and 2q + 3 configurations; SMC-FCS rejection-sampling diagnostics) | Table 2, Fig. 2 (left) |
| 5. Extended scenarios, 200 replicates | Table 3, Fig. 3 |
| 6. Finite configurations against a dense search | Table 4 |
| 7. Discrimination (C-index) | Fig. 7 |

The code in sections 2–3 is identical to the code that produced the paper's results, and so are the seeds (replicate *r* uses seed 10,000 + *r*). A full run reproduces the reported numbers. Model fits are refined by Newton–Raphson steps with analytic derivatives, so they do not depend on optimiser stopping rules or hardware.

**Data.** SUPPORT2 is public (UCI Machine Learning Repository, doi:10.3886/ICPSR02957.v2); it is loaded through the `SurvSet` package.

**Run time.** `QUICK = True` runs a handful of replicates of everything, which takes about 10–20 minutes. `QUICK = False` runs the full study, roughly 20–30 CPU-hours; replicates can be distributed with `START`/`REPS`. Results are appended to CSV files after every replicate, so a run can be interrupted and resumed.

**Requirements.** Python ≥ 3.10; `numpy pandas scipy jax jaxlib scikit-learn scikit-survival lifelines matplotlib SurvSet` (see `requirements.txt`).

## 1. Settings

In [ ]:
QUICK = True
OUTDIR = "results_s2"
MAIN_REPS = 3 if QUICK else 1000        # per mechanism M1-M4
EXT_REPS = 1 if QUICK else 200          # per extended scenario
DENSE_REPS = 1 if QUICK else 20         # per mechanism
CIDX_REPS = 1 if QUICK else 20
EXT_SCENARIOS = ["n500", "n1000", "misslow", "misshigh", "self05", "self10", "self20", "quad"]
import os; os.makedirs(OUTDIR, exist_ok=True)
os.environ.setdefault("OMP_NUM_THREADS", "1")

## 2. Core library
Royston–Parmar model (analytic derivatives, Newton refinement), logistic regression, Rubin's rules, SMC-FCS imputation, regression calibration and the envelope search (2*q* + 3 configurations, adaptive vertices, dense search).

In [ ]:
"""
hppsm_v3.py - core library for the HPPSM analyses (Draft 7).

Contents
  * RoystonParmarPH : flexible parametric PH model (JAX likelihood), fitted by L-BFGS-B and then
                      refined by Newton-Raphson steps to a gradient norm below 1e-8, so the maximum
                      likelihood estimate does not depend on optimiser stopping rules.
  * logit_fit       : unpenalised logistic regression (coefficients and SEs per raw unit).
  * rubin           : Rubin's rules.
  * smcfcs_rp       : substantive-model-compatible FCS imputation for the RP model (Bartlett et al. 2015).
  * regression_calibration, error_variance_bound : interval-summarised covariates.
  * Envelope        : union of core intervals over mechanism configurations, computed by
                        - vertex_2q3  : reference, each variable at +/-dmax alone, all +dmax, all -dmax
                        - adaptive    : the above plus, for every endpoint of every target, the vertex
                                        whose signs are read off the single-variable runs (exact when
                                        each endpoint is monotone in each coordinate; Proposition 5)
                        - dense       : all 2^q vertices plus random points in the box (verification)
"""
import itertools
import numpy as np, pandas as pd
import jax, jax.numpy as jnp
from scipy.optimize import minimize
from scipy.stats import t as tdist
jax.config.update("jax_enable_x64", True)
Q95 = 1.959963984540054


# ------------------------------------------------------------------ Royston-Parmar model
def _standardise(A, mu=None, sd=None):
    A = np.asarray(A, float)
    if mu is None:
        mu = A.mean(0); sd = A.std(0); sd = np.where(sd > 0, sd, 1.0)
    return (A - mu) / sd, mu, sd


def _pos3(x):
    return jnp.where(x > 0, x, 0.0) ** 3


def _dpos3(x):
    return 3.0 * jnp.where(x > 0, x, 0.0) ** 2


def rp_basis(u, knots):
    kmin, kmax = knots[0], knots[-1]
    cols = [jnp.ones_like(u), u]
    for k in knots[1:-1]:
        lam = (kmax - k) / (kmax - kmin)
        cols.append(_pos3(u - k) - lam * _pos3(u - kmin) - (1 - lam) * _pos3(u - kmax))
    return jnp.stack(cols, axis=-1)


def rp_dbasis(u, knots):
    kmin, kmax = knots[0], knots[-1]
    cols = [jnp.zeros_like(u), jnp.ones_like(u)]
    for k in knots[1:-1]:
        lam = (kmax - k) / (kmax - kmin)
        cols.append(_dpos3(u - k) - lam * _dpos3(u - kmin) - (1 - lam) * _dpos3(u - kmax))
    return jnp.stack(cols, axis=-1)


def _rp_ll(th, u, d, Zs, Xs, knots, ng):
    g, bz, bx = th[:ng], th[ng:ng + Zs.shape[1]], th[ng + Zs.shape[1]:]
    s = rp_basis(u, knots) @ g; ds = rp_dbasis(u, knots) @ g
    eta = Zs @ bz + Xs @ bx
    return jnp.sum(d * (jnp.log(jnp.clip(ds, 1e-10)) - u + s + eta) - jnp.exp(s + eta))


_rp_nll = jax.jit(lambda *a: -_rp_ll(*a), static_argnums=6)
_rp_grad = jax.jit(jax.grad(lambda *a: -_rp_ll(*a)), static_argnums=6)
_rp_hess = jax.jit(jax.hessian(_rp_ll), static_argnums=6)


def _newton_polish(f, g, h, th, tol=1e-8, max_iter=8):
    """Newton-Raphson refinement of a minimiser of f (h = Hessian of -f) with step halving."""
    fv = f(th)
    for _ in range(max_iter):
        gr = g(th)
        if np.max(np.abs(gr)) < tol:
            break
        H = h(th)
        try:
            step = np.linalg.solve(H, gr)          # H is the log-likelihood Hessian (negative definite)
        except np.linalg.LinAlgError:
            break
        lam = 1.0
        while lam > 1e-4:
            new = th + lam * step
            fn = f(new)
            if np.isfinite(fn) and fn <= fv + 1e-12:
                th, fv = new, fn; break
            lam /= 2
        else:
            break
    return th


def _np_basis(u, knots):
    knots = np.asarray(knots); kmin, kmax = knots[0], knots[-1]
    p3 = lambda x: np.where(x > 0, x, 0.0) ** 3; d3 = lambda x: 3.0 * np.where(x > 0, x, 0.0) ** 2
    B = [np.ones_like(u), u]; dB = [np.zeros_like(u), np.ones_like(u)]
    for k in knots[1:-1]:
        lam = (kmax - k) / (kmax - kmin)
        B.append(p3(u - k) - lam * p3(u - kmin) - (1 - lam) * p3(u - kmax))
        dB.append(d3(u - k) - lam * d3(u - kmin) - (1 - lam) * d3(u - kmax))
    return np.column_stack(B), np.column_stack(dB)


class _RPLik:
    """Analytic log-likelihood, gradient and Hessian of the Royston-Parmar PH model (numpy)."""

    def __init__(self, u, d, Zs, Xs, knots):
        self.B, self.dB = _np_basis(u, knots); self.u, self.d = u, d
        self.D = np.hstack([self.B, Zs, Xs]); self.ng = self.B.shape[1]

    def _parts(self, th):
        ds = self.dB @ th[:self.ng]; lin = self.D @ th
        return ds, lin

    def nll(self, th):
        ds, lin = self._parts(th)
        if np.any(ds <= 0):
            return np.inf
        return -float(np.sum(self.d * (np.log(ds) - self.u + lin) - np.exp(np.minimum(lin, 700))))

    def grad(self, th):                       # gradient of the negative log-likelihood
        ds, lin = self._parts(th); mu = np.exp(np.minimum(lin, 700))
        g = self.D.T @ (self.d - mu)
        g[:self.ng] += self.dB.T @ (self.d / np.clip(ds, 1e-10, None))
        return -g

    def hess(self, th):                       # Hessian of the log-likelihood
        ds, lin = self._parts(th); mu = np.exp(np.minimum(lin, 700))
        H = -(self.D.T * mu) @ self.D
        w = self.d / np.clip(ds, 1e-10, None) ** 2
        H[:self.ng, :self.ng] -= (self.dB.T * w) @ self.dB
        return H


class RoystonParmarPH:
    """log H(t|z,x) = s(log t) + z'b_z + x'b_x; s a natural cubic spline with `df` d.f.
    Covariates are standardised internally (theta and cov on the standardised scale)."""

    def __init__(self, df=3):
        self.df = df

    def fit(self, t, event, Z, X, theta0=None):
        t = np.asarray(t, float); event = np.asarray(event, float); u = np.log(t)
        self.knots = jnp.asarray(np.quantile(u[event == 1], np.linspace(0, 1, self.df + 1)))
        Zs, self.z_mu, self.z_sd = _standardise(Z)
        Xs, self.x_mu, self.x_sd = _standardise(X)
        Zs, Xs = np.ascontiguousarray(Zs), np.ascontiguousarray(Xs)
        ng = self.df + 1
        self.ix_g = np.arange(ng); self.ix_z = np.arange(ng, ng + Zs.shape[1])
        self.ix_x = np.arange(ng + Zs.shape[1], ng + Zs.shape[1] + Xs.shape[1])
        warm = theta0 is not None
        if theta0 is None:
            theta0 = np.zeros(ng + Zs.shape[1] + Xs.shape[1]); theta0[0] = np.log(event.sum() / t.sum()); theta0[1] = 1.0
        L = _RPLik(u, event, Zs, Xs, np.asarray(self.knots))
        f, g, h = L.nll, L.grad, L.hess
        th = None
        if warm:                                   # warm start: Newton-Raphson directly (quadratic convergence)
            th = _newton_polish(f, g, h, np.asarray(theta0, float), max_iter=25)
            if not np.all(np.isfinite(th)) or np.max(np.abs(g(th))) > 1e-6:
                th = None
        if th is None:
            res = minimize(f, theta0, jac=g, method="L-BFGS-B", options={"maxiter": 5000, "gtol": 1e-7})
            th = _newton_polish(f, g, h, res.x)
        self.theta = th
        self.cov = np.linalg.inv(-h(self.theta))
        self.loglik = -f(self.theta)
        return self

    def coef(self, block, j):
        k = (self.ix_z if block == "z" else self.ix_x)[j]
        s = (self.z_sd if block == "z" else self.x_sd)[j]
        return self.theta[k] / s, np.sqrt(self.cov[k, k]) / s

    def lp(self, Z, X):
        return ((np.asarray(Z) - self.z_mu) / self.z_sd) @ self.theta[self.ix_z] + \
               ((np.asarray(X) - self.x_mu) / self.x_sd) @ self.theta[self.ix_x]


# ------------------------------------------------------------------ logistic regression
def logit_fit(y, X):
    """Unpenalised logistic regression by Newton-Raphson. Returns (coefficients, SEs) per raw unit
    (intercept excluded) and the intercept as a third element."""
    X = np.asarray(X, float); y = np.asarray(y, float)
    mu, s = X.mean(0), X.std(0); s = np.where(s > 0, s, 1.0)
    Xd = np.hstack([np.ones((len(y), 1)), (X - mu) / s])
    b = np.zeros(Xd.shape[1]); b[0] = np.log(y.mean() / (1 - y.mean()))
    for _ in range(50):
        eta = Xd @ b; p = 1 / (1 + np.exp(-eta))
        W = p * (1 - p); H = Xd.T @ (Xd * W[:, None]); gr = Xd.T @ (y - p)
        step = np.linalg.solve(H, gr); b = b + step
        if np.max(np.abs(step)) < 1e-10:
            break
    p = 1 / (1 + np.exp(-(Xd @ b)))
    cov = np.linalg.inv(Xd.T @ (Xd * (p * (1 - p))[:, None]))
    coef = b[1:] / s
    return coef, np.sqrt(np.diag(cov))[1:] / s, b[0] - np.sum(b[1:] * mu / s)


# ------------------------------------------------------------------ pooling
def rubin(ests, vars_, alpha=0.05):
    ests, vars_ = np.asarray(ests, float), np.asarray(vars_, float)
    m = len(ests); qb = ests.mean(0); W = vars_.mean(0)
    if m == 1:
        z = tdist.ppf(1 - alpha / 2, 1e6); return qb, qb - z * np.sqrt(W), qb + z * np.sqrt(W)
    B = ests.var(0, ddof=1); Tv = W + (1 + 1 / m) * B
    with np.errstate(divide="ignore", invalid="ignore"):
        dfr = np.where(B > 0, (m - 1) * (1 + W / ((1 + 1 / m) * B)) ** 2, 1e6)
    tq = tdist.ppf(1 - alpha / 2, dfr)
    return qb, qb - tq * np.sqrt(Tv), qb + tq * np.sqrt(Tv)


# ------------------------------------------------------------------ interval-summarised covariates
def error_variance_bound(lo, hi, n):
    """Bound (1): Var(U) <= (n-2) w^2 / (4 n^2), interior measurements treated as independent."""
    return np.maximum(n - 2, 0) * (hi - lo) ** 2 / (4 * np.maximum(n, 1) ** 2)


def regression_calibration(W, Z, sigma_u2):
    """E[X*|W,Z] under W = X* + U with U independent of (X*, Z, Y), E U = 0, Var U = diag(sigma_u2);
    computed from first and second moments. Error variance capped at 80% of Var(W)."""
    V = np.hstack([W, Z]); mu = V.mean(0); S = np.cov(V, rowvar=False)
    q = W.shape[1]
    sigma_u2 = np.minimum(sigma_u2, 0.8 * np.diag(S)[:q])
    C = S[:q, :].copy(); C[:, :q] -= np.diag(sigma_u2)
    return W.mean(0) + (V - mu) @ np.linalg.solve(S, C.T)


# ------------------------------------------------------------------ SMC-FCS for the RP model
def truncnorm_draw(rng, mu, sd, lo, hi):
    """Exact draws from N(mu, sd^2) truncated to [lo, hi] by inversion, computed in the tail that keeps
    precision (no clipping, so no artificial mass at the limits)."""
    from scipy.special import ndtr, ndtri
    mu = np.asarray(mu, float); a = (lo - mu) / sd; b = (hi - mu) / sd
    u = rng.uniform(size=mu.shape)
    x = np.empty_like(mu)
    up = a > 0                                     # whole interval in the upper tail: invert the survival function
    Ga, Gb = ndtr(-a[up]), ndtr(-b[up])
    x[up] = -ndtri(Gb + u[up] * (Ga - Gb))
    dn = ~up
    Fa, Fb = ndtr(a[dn]), ndtr(b[dn])
    x[dn] = ndtri(Fa + u[dn] * (Fb - Fa))
    return np.clip(mu + sd * x, lo, hi)            # clip only guards against rounding at the limits


def smcfcs_rp(T, E, Z, Xmiss, lo, hi, n_cycles=8, m=3, seed=0, max_tries=1000, theta0=None, return_diag=False):
    """Substantive-model-compatible FCS imputation (Bartlett et al. 2015) for the Royston-Parmar model.
    Proposals come from the normal linear covariate model (parameters drawn from their posterior) truncated
    to the variable's support [lo, hi]; they are accepted with probability exp(-H) if censored and
    H exp(1 - H) if died, with H the cumulative hazard under a parameter draw from the current model fit.
    Values not accepted after `max_tries` proposals keep their value from the previous cycle; the number of
    such values is reported in the diagnostics (return_diag=True)."""
    rng = np.random.default_rng(seed)
    Xm = np.asarray(Xmiss, float); M = np.isnan(Xm); n, q = Xm.shape
    miss_cols = [j for j in range(q) if M[:, j].any()]
    out = []; diag = dict(draws=0, not_accepted=0, proposals=0, final_cycle_not_accepted=0)
    for k in range(m):
        X = Xm.copy()
        for j in range(q):
            X[M[:, j], j] = np.nanmean(Xm[:, j])
        th_prev = theta0
        for cyc in range(n_cycles):
            rp = RoystonParmarPH(df=3).fit(T, E, Z, X, theta0=th_prev); th_prev = rp.theta
            th = rng.multivariate_normal(rp.theta, rp.cov)
            s = np.asarray(rp_basis(jnp.asarray(np.log(T)), rp.knots)) @ th[rp.ix_g]
            eta_z = ((Z - rp.z_mu) / rp.z_sd) @ th[rp.ix_z]
            bx = th[rp.ix_x] / rp.x_sd
            for j in miss_cols:
                rows = np.where(M[:, j])[0]
                D = np.hstack([np.ones((n, 1)), Z, np.delete(X, j, axis=1)])
                obs = ~M[:, j]; Do, yo = D[obs], X[obs, j]
                XtX = Do.T @ Do + 1e-8 * np.eye(D.shape[1])
                bh = np.linalg.solve(XtX, Do.T @ yo)
                res = yo - Do @ bh; dfree = max(obs.sum() - D.shape[1], 1)
                s2 = (res @ res) / rng.chisquare(dfree)
                bd = rng.multivariate_normal(bh, s2 * np.linalg.inv(XtX))
                mu = D[rows] @ bd
                eta_other = eta_z[rows] + (X[rows] - rp.x_mu) @ bx - (X[rows, j] - rp.x_mu[j]) * bx[j]
                pending = np.ones(len(rows), bool); draw = X[rows, j].copy()     # fall-back: previous value
                for _ in range(max_tries):
                    if not pending.any():
                        break
                    idx = np.where(pending)[0]
                    prop = truncnorm_draw(rng, mu[idx], np.sqrt(s2), lo[j], hi[j])
                    H = np.exp(np.minimum(s[rows[idx]] + eta_other[idx] + (prop - rp.x_mu[j]) * bx[j], 700.0))
                    acc_p = np.where(E[rows[idx]] == 1, H * np.exp(1.0 - H), np.exp(-H))
                    acc = rng.uniform(size=len(idx)) < acc_p
                    diag["proposals"] += len(idx)
                    draw[idx[acc]] = prop[acc]; pending[idx[acc]] = False
                X[rows, j] = draw
                diag["draws"] += len(rows); diag["not_accepted"] += int(pending.sum())
                if cyc == n_cycles - 1:
                    diag["final_cycle_not_accepted"] += int(pending.sum())
        out.append(X.copy())
    return (out, diag) if return_diag else out


# ------------------------------------------------------------------ envelope over mechanism parameters
class Envelope:
    """Union of core intervals over configurations of the MNAR shift delta (one coordinate per heavily
    missing variable). `core(delta_vector) -> (est, lo, hi)` arrays over the targets. Results are cached,
    so the different search strategies share their evaluations."""

    def __init__(self, core, q):
        self.core, self.q, self.cache = core, q, {}

    def eval(self, d):
        key = tuple(np.round(np.asarray(d, float), 10))
        if key not in self.cache:
            self.cache[key] = self.core(np.asarray(d, float))
        return self.cache[key]

    def _union(self, keys):
        lo = np.min([self.cache[k][1] for k in keys], 0); hi = np.max([self.cache[k][2] for k in keys], 0)
        return lo, hi

    def vertex_2q3(self, dmax):
        q = self.q; cfg = [np.zeros(q)]
        for j in range(q):
            for s in (1, -1):
                d = np.zeros(q); d[j] = s * dmax; cfg.append(d)
        cfg += [np.full(q, dmax), np.full(q, -dmax)]
        keys = [tuple(np.round(c, 10)) for c in cfg]
        for c in cfg:
            self.eval(c)
        return self._union(keys), len(set(keys))

    def adaptive(self, dmax):
        """2q+3 configurations plus the sign-matched vertex for each endpoint of each target."""
        (_, _), _ = self.vertex_2q3(dmax)
        q = self.q
        plus = [self.eval(np.eye(q)[j] * dmax) for j in range(q)]
        minus = [self.eval(-np.eye(q)[j] * dmax) for j in range(q)]
        n_t = len(plus[0][0]) if q else 0
        verts = []
        for t in range(n_t):
            s_lo = np.array([np.sign(plus[j][1][t] - minus[j][1][t]) for j in range(q)])
            s_hi = np.array([np.sign(plus[j][2][t] - minus[j][2][t]) for j in range(q)])
            verts.append(-dmax * np.where(s_lo == 0, 1, s_lo))     # minimises the lower endpoint
            verts.append(dmax * np.where(s_hi == 0, 1, s_hi))      # maximises the upper endpoint
        cfg = [np.zeros(q)] + [np.eye(q)[j] * s * dmax for j in range(q) for s in (1, -1)] + \
              [np.full(q, dmax), np.full(q, -dmax)] + verts
        keys = [tuple(np.round(c, 10)) for c in cfg]
        for c in cfg:
            self.eval(c)
        return self._union(keys), len(set(keys))

    def vertices(self, dmax):
        """Reference configuration and all 2^q vertices of the box (full vertex enumeration).
        Exact for the interval hull whenever each endpoint is monotone in each coordinate (Proposition 5)."""
        q = self.q
        cfg = [np.zeros(q)] + [np.array(v, float) * dmax for v in itertools.product((-1, 1), repeat=q)]
        keys = [tuple(np.round(c, 10)) for c in cfg]
        for c in cfg:
            self.eval(c)
        return self._union(keys), len(set(keys))

    def dense(self, dmax, n_random=100, rng=None):
        """All 2^q vertices plus n_random uniform points in the box (verification only)."""
        q = self.q; rng = rng or np.random.default_rng(0)
        cfg = [np.array(v) * dmax for v in itertools.product((-1, 1), repeat=q)] + \
              [rng.uniform(-dmax, dmax, q) for _ in range(n_random)] + [np.zeros(q)]
        keys = [tuple(np.round(c, 10)) for c in cfg]
        for c in cfg:
            self.eval(c)
        return self._union(keys), len(set(keys))


def coverage_summary(df, oracle="Oracle"):
    """Mean / lowest coverage across terms, Monte Carlo SE of the mean, median width relative to oracle."""
    d = df.copy(); d["cover"] = (d.lo <= d.truth) & (d.truth <= d.hi); d["w"] = d.hi - d.lo
    ow = d[d.method == oracle].groupby("term").w.mean()
    g = d.groupby(["method", "term"]).agg(cvg=("cover", "mean"), w=("w", "mean"), R=("cover", "size")).reset_index()
    g["mcse"] = np.sqrt(g.cvg * (1 - g.cvg) / g.R)
    g["wr"] = g.w / g.term.map(ow)
    s = g.groupby("method").agg(mean_coverage=("cvg", "mean"), lowest_coverage=("cvg", "min"),
                                width_vs_oracle=("wr", "median"), reps=("R", "min"))
    return s, g

## 3. SUPPORT2 plasmode: truth, data generation and one replicate
Truth model, simulation of survival times, coarsening mechanisms (M1–M4 and extended scenarios), and the replicate functions used in the paper.

In [ ]:
import os, sys, time, json
"""
s2_study.py - SUPPORT2 plasmode simulation for Draft 7.

Scenarios (all with 3,000 patients unless stated; replicate r uses seed 10_000 + r for the data):
  main       M1-M4 (as in the paper), 1,000 replicates
  extended   n500, n1000            : smaller samples (M1)
             misslow, misshigh      : missing rates x0.5 and x1.5 (M1)
             self05, self10, self20 : self-masking MNAR - missingness also depends on the variable's own
                                      true value with strength lambda = 0.5, 1, 2 (logit per SD), on top of M1
             quad                   : outcome model misspecified (true log hazard has a quadratic BUN term;
                                      the analysis model is linear); estimand = pseudo-true coefficients
  dense      verification of the finite-configuration union against a dense search (M1-M4)
Methods: Oracle, Fill-in Wald, Complete case, MI (chained equations), MI (SMC-FCS),
         HPPSM v2 (2q+3 configurations; as in Draft 6), HPPSM v2 (adaptive vertices; primary),
         and in extended scenarios HPPSM with dmax = 0.25 and 1.0.
"""
import os, sys, time, json, argparse
import numpy as np, pandas as pd
import jax.numpy as jnp
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

LABS = pd.DataFrame([
    ("alb", "id", 1.0, 5.5, 3.5), ("pafi", "id", 50.0, 600.0, 333.3), ("bili", "log", 0.1, 30.0, 1.01),
    ("crea", "log", 0.3, 12.0, 1.01), ("bun", "log", 3.0, 150.0, 6.51), ("wblc", "log", 0.5, 60.0, 9.0),
    ("urine", "log1p", 0.0, 8000., 2502.), ("ph", "id", 6.9, 7.7, 7.40), ("glucose", "log", 40.0, 600.0, 100.0),
], columns=["var", "tf", "lo", "hi", "fill"]).set_index("var")
TF = {"id": lambda v: v, "log": np.log, "log1p": np.log1p}
TERMS_X = ["bun", "crea", "alb", "glucose"]; TERMS_Z = ["age", "scoma"]; TERMS = TERMS_X + TERMS_Z
LO_SUP = np.array([TF[LABS.loc[v, "tf"]](LABS.loc[v, "lo"]) for v in LABS.index])
HI_SUP = np.array([TF[LABS.loc[v, "tf"]](LABS.loc[v, "hi"]) for v in LABS.index])
MECH = {"M1": (0.75, False), "M2": (1.5, False), "M3": (0.0, False), "M4": (1.0, True)}
EXT = {  # name: (gamma, ydep, n, miss_scale, self_lambda, quad)
    "n500": (0.75, False, 500, 1.0, 0.0, 0.0), "n1000": (0.75, False, 1000, 1.0, 0.0, 0.0),
    "misslow": (0.75, False, 3000, 0.5, 0.0, 0.0), "misshigh": (0.75, False, 3000, 1.5, 0.0, 0.0),
    "self05": (0.75, False, 3000, 1.0, 0.5, 0.0), "self10": (0.75, False, 3000, 1.0, 1.0, 0.0),
    "self20": (0.75, False, 3000, 1.0, 2.0, 0.0), "quad": (0.75, False, 3000, 1.0, 0.0, 0.15),
}
DMAX = 0.5


def build_truth(seed=11):
    from SurvSet.data import SurvLoader
    raw, _ = SurvLoader().load_dataset("support2").values()
    d = raw.rename(columns=lambda c: c.replace("num_", "", 1).replace("fac_", "", 1)).rename(columns={"num_co": "num.co"})
    d = d.loc[:, ~d.columns.duplicated()]
    d = d.dropna(subset=["age", "num.co", "scoma", "hday", "meanbp", "hrt", "resp", "temp", "sod", "sex",
                         "dzgroup", "diabetes", "dementia", "ca"]).reset_index(drop=True)
    Z = pd.DataFrame(index=d.index)
    Z["age"] = d["age"]; Z["age_sq"] = ((d["age"] - 65) / 10) ** 2
    Z["num_co"] = d["num.co"]; Z["scoma"] = d["scoma"]; Z["log_hday"] = np.log1p(d["hday"])
    for v, c in [("meanbp", 80), ("hrt", 90), ("resp", 20), ("temp", 37), ("sod", 138)]:
        Z[v] = d[v]; Z[v + "_sq"] = ((d[v] - c) / d[v].std()) ** 2
    Z["male"] = (d["sex"].astype(str) == "male").astype(float)
    Z["diabetes"] = d["diabetes"].astype(float); Z["dementia"] = d["dementia"].astype(float)
    ca = d["ca"].astype(str); Z["ca_yes"] = (ca == "yes").astype(float); Z["ca_metastatic"] = (ca == "metastatic").astype(float)
    dz = pd.get_dummies(d["dzgroup"].astype(str), prefix="dz", dtype=float)
    Z = Z.join(dz.drop(columns=["dz_ARF/MOSF w/Sepsis"], errors="ignore"))
    Xobs = pd.DataFrame({v: TF[r.tf](d[v].clip(r.lo, r.hi)) for v, r in LABS.iterrows()})
    miss_rate = Xobs.isna().mean()
    imp = IterativeImputer(random_state=seed, sample_posterior=True, max_iter=10)
    Xtrue = pd.DataFrame(imp.fit_transform(np.hstack([Z.values, Xobs.values]))[:, Z.shape[1]:], columns=Xobs.columns)
    for v, r in LABS.iterrows():
        Xtrue[v] = Xtrue[v].clip(TF[r.tf](r.lo), TF[r.tf](r.hi))
    T, E = d["time"].astype(float).clip(lower=0.5).values, d["event"].values.astype(float)
    m = RoystonParmarPH(df=3).fit(T, E, Z.values, Xtrue.values)
    theta = m.theta.copy()
    for v, b in {"bun": 0.35, "crea": 0.35, "alb": -0.35, "glucose": 0.2}.items():
        theta[m.ix_x[list(Xtrue.columns).index(v)]] = b
    tc = {**{v: theta[m.ix_x[list(Xtrue.columns).index(v)]] / m.x_sd[list(Xtrue.columns).index(v)] for v in TERMS_X},
          **{v: theta[m.ix_z[list(Z.columns).index(v)]] / m.z_sd[list(Z.columns).index(v)] for v in TERMS_Z}}
    return dict(model=m, theta=theta, Z=Z, Xtrue=Xtrue, miss_rate=miss_rate, true_coef=tc)


def simulate_times(tr, rows, rng, quad=0.0):
    m, th = tr["model"], tr["theta"]
    Zs = (tr["Z"].values[rows] - m.z_mu) / m.z_sd
    Xs = (tr["Xtrue"].values[rows] - m.x_mu) / m.x_sd
    eta = Zs @ th[m.ix_z] + Xs @ th[m.ix_x]
    if quad:
        jb = list(tr["Xtrue"].columns).index("bun"); eta = eta + quad * (Xs[:, jb] ** 2 - 1)
    target = np.log(-np.log(rng.uniform(size=len(rows)))) - eta
    g = th[m.ix_g]
    lo, hi = np.full(len(rows), np.log(0.5)), np.full(len(rows), np.log(20000.0))
    s = lambda u: np.asarray(rp_basis(jnp.asarray(u), m.knots)) @ g
    for _ in range(60):
        mid = (lo + hi) / 2; up = s(mid) < target
        lo, hi = np.where(up, mid, lo), np.where(up, hi, mid)
    t = np.exp((lo + hi) / 2)
    c = rng.uniform(180, 2000, len(rows))
    return np.minimum(t, c), (t <= c).astype(float), eta, t


def coarsen(tr, rows, driver, gamma, rng, miss_scale=1.0, self_lambda=0.0):
    Xt = tr["Xtrue"].iloc[rows].reset_index(drop=True)
    sev = (driver - driver.mean()) / driver.std()
    miss = pd.DataFrame(index=Xt.index)
    for v in Xt.columns:
        pm = min(float(tr["miss_rate"][v]) * miss_scale, 0.85)
        if float(tr["miss_rate"][v]) <= 0.005:
            miss[v] = False; continue
        own = (Xt[v] - Xt[v].mean()) / Xt[v].std()
        lin = -gamma * sev - self_lambda * own.values        # high values (abnormal) are measured more often
        a = np.log(pm / (1 - pm))
        for _ in range(30):
            p = 1 / (1 + np.exp(-(a + lin)))
            a += np.log(pm / (1 - pm)) - np.log(p.mean() / (1 - p.mean()))
        miss[v] = rng.uniform(size=len(Xt)) < 1 / (1 + np.exp(-(a + lin)))
    Xref = Xt.copy()
    for v in Xt.columns:
        Xref.loc[miss[v].values, v] = TF[LABS.loc[v, "tf"]](LABS.loc[v, "fill"])
    return Xt, Xref, miss


def generate(tr, seed, n, gamma, ydep, miss_scale=1.0, self_lambda=0.0, quad=0.0):
    rng = np.random.default_rng(seed)
    rows = rng.choice(len(tr["Z"]), n, replace=False)
    T, E, eta, t_true = simulate_times(tr, rows, rng, quad)
    Xt, Xref, miss = coarsen(tr, rows, -np.log(t_true) if ydep else eta, gamma, rng, miss_scale, self_lambda)
    Z = tr["Z"].iloc[rows].reset_index(drop=True); Z = Z.loc[:, Z.std() > 0]
    return rng, rows, T, E, Xt, Xref, miss, Z


def rp_terms(T, E, Z, X, theta0=None, return_theta=False):
    m = RoystonParmarPH(df=3).fit(T, E, Z.values, X, theta0=theta0)
    zn, xn = list(Z.columns), list(LABS.index)
    est = [m.coef("x", xn.index(v)) for v in TERMS_X] + [m.coef("z", zn.index(v)) for v in TERMS_Z]
    res = np.array([e for e, _ in est]), np.array([s for _, s in est])
    return (*res, m.theta) if return_theta else res


def wald(e, s):
    return e, e - Q95 * s, e + Q95 * s


def smc_envelope(T, E, Z, Xt, miss, rep, m_smc=3, theta0=None, diag=None):
    """SMC-FCS imputations (m = 3, 8 cycles, at most 1,000 proposals per value) and the envelope object over the
    heavily missing variables. theta0 (optional) is a starting value for the model fits; fits are refined to the
    exact maximum, so it affects speed only. If `diag` is a dict, rejection-sampling diagnostics are stored in it."""
    Xm = Xt.mask(miss); sd = Xm.std()
    imps, dg = smcfcs_rp(T, E, Z.values, Xm.values, LO_SUP, HI_SUP, m=m_smc, seed=7000 + rep, theta0=theta0, return_diag=True)
    if diag is not None:
        diag.update(dg)
    heavy = [v for v in LABS.index if miss[v].mean() > 0.05]
    cols = list(LABS.index)
    warm = {k: theta0 for k in range(len(imps))} if theta0 is not None else {}

    def core(d):
        ests, vars_ = [], []
        for k, A in enumerate(imps):
            A2 = A.copy()
            for j, v in enumerate(heavy):
                if d[j]:
                    A2[miss[v].values, cols.index(v)] += d[j] * sd[v]
            e, s, th = rp_terms(T, E, Z, A2, theta0=warm.get(k), return_theta=True)
            if k not in warm or warm[k] is theta0:
                warm[k] = th                           # warm start from the reference-configuration fit
            ests.append(e); vars_.append(s ** 2)
        return rubin(ests, vars_)
    return Envelope(core, len(heavy)), imps, heavy


def run_rep(tr, scen, rep, truth=None):
    if scen in MECH:
        gamma, ydep = MECH[scen]; n, ms, lam, quad = 3000, 1.0, 0.0, 0.0
    else:
        gamma, ydep, n, ms, lam, quad = EXT[scen]
    truth = truth or tr["true_coef"]
    _, _, T, E, Xt, Xref, miss, Z = generate(tr, 10_000 + rep, n, gamma, ydep, ms, lam, quad)
    out = {}
    e0, s0, th0 = rp_terms(T, E, Z, Xt.values, return_theta=True); out["Oracle"] = wald(e0, s0)
    env, imps, heavy = smc_envelope(T, E, Z, Xt, miss, rep, theta0=th0)
    out["MI, SMC-FCS"] = env.eval(np.zeros(env.q))
    (lo, hi), _ = env.vertex_2q3(DMAX); out["HPPSM v2 (2q+3 configurations)"] = (out["MI, SMC-FCS"][0], lo, hi)
    (lo, hi), ncfg = env.adaptive(DMAX); out["HPPSM v2 (proposed)"] = (out["MI, SMC-FCS"][0], lo, hi)
    if scen not in MECH:
        for dm in (0.25, 1.0):
            (lo, hi), _ = env.adaptive(dm); out[f"HPPSM v2 (dmax = {dm})"] = (out["MI, SMC-FCS"][0], lo, hi)
    out["Fill-in Wald"] = wald(*rp_terms(T, E, Z, Xref.values, theta0=th0))
    cc = ~miss[TERMS_X].any(axis=1).values
    if cc.sum() > 100:
        Zc = Z[cc].loc[:, Z[cc].std() > 0]
        try:
            out["Complete case"] = wald(*rp_terms(T[cc], E[cc], Zc, Xt.values[cc]))
        except Exception:
            pass
    # chained-equation MI (a comparator known to be biased here) is computed for the first 200 replicates of
    # the main mechanisms only, because it accounts for about half of the computing time
    if scen not in MECH or rep >= 200:
        k_mi = 0
    else:
        k_mi = 5
    from lifelines import NelsonAalenFitter
    na = NelsonAalenFitter().fit(T, E).cumulative_hazard_at_times(T).values
    Xm = Xt.mask(miss); ests, vars_ = [], []
    for k in range(k_mi):
        A = IterativeImputer(random_state=1000 * rep + k, sample_posterior=True, max_iter=5).fit_transform(
            np.hstack([Z.values, E[:, None], na[:, None], Xm.values]))[:, Z.shape[1] + 2:]
        e, s = rp_terms(T, E, Z, A, theta0=th0); ests.append(e); vars_.append(s ** 2)
    if k_mi:
        out["MI, chained equations"] = rubin(ests, vars_)
    rows = pd.DataFrame([dict(rep=rep, method=k, term=t, est=e[i], lo=l[i], hi=h[i], truth=truth[t])
                         for k, (e, l, h) in out.items() for i, t in enumerate(TERMS)])
    # empirical MNAR shift: mean (true - imputed) among missing values, in SD units of the observed values
    cols = list(LABS.index)
    dl = {v: float(np.mean([np.mean(Xt[v].values[miss[v].values] - A[miss[v].values, cols.index(v)]) for A in imps])
                   / Xm[v].std()) for v in heavy}
    info = dict(rep=rep, n_configs=ncfg, cc=int(cc.sum()), **{f"delta_{v}": dl[v] for v in heavy})
    return rows, info


def vertex_rep(tr, scen, rep, truth=None):
    """Full vertex enumeration (primary in Draft 8) for one replicate; the data, imputations and reference
    analysis are identical to run_rep, so the rows can be merged with its output."""
    if scen in MECH:
        gamma, ydep = MECH[scen]; n, ms, lam, quad = 3000, 1.0, 0.0, 0.0
    else:
        gamma, ydep, n, ms, lam, quad = EXT[scen]
    truth = truth or tr["true_coef"]
    _, _, T, E, Xt, Xref, miss, Z = generate(tr, 10_000 + rep, n, gamma, ydep, ms, lam, quad)
    th0 = rp_terms(T, E, Z, Xt.values, return_theta=True)[2]
    env, imps, heavy = smc_envelope(T, E, Z, Xt, miss, rep, theta0=th0)
    e0 = env.eval(np.zeros(env.q))[0]
    (lo, hi), nv = env.vertices(DMAX)
    (la, ha), _ = env.adaptive(DMAX)
    out = {"HPPSM v2 (vertices only)": (e0, lo, hi), "HPPSM v2 (adaptive, check)": (e0, la, ha),
           "HPPSM v2 (vertices + axes)": (e0, np.minimum(lo, la), np.maximum(hi, ha))}   # primary in Draft 8
    nv = len(env.cache)
    return pd.DataFrame([dict(rep=rep, method=k, term=t, est=e[i], lo=l[i], hi=h[i], truth=truth[t], n_configs=nv)
                         for k, (e, l, h) in out.items() for i, t in enumerate(TERMS)])


def smc_rep(tr, scen, rep, truth=None):
    """All SMC-FCS-based methods for one replicate (Draft 9: exact truncated proposals, explicit fall-back).
    Data are identical to run_rep; the non-SMC methods (oracle, fill-in, complete case, chained-equation MI)
    do not depend on the sampler and are taken from run_rep."""
    if scen in MECH:
        gamma, ydep = MECH[scen]; n, ms, lam, quad = 3000, 1.0, 0.0, 0.0
    else:
        gamma, ydep, n, ms, lam, quad = EXT[scen]
    truth = truth or tr["true_coef"]
    _, _, T, E, Xt, Xref, miss, Z = generate(tr, 10_000 + rep, n, gamma, ydep, ms, lam, quad)
    th0 = rp_terms(T, E, Z, Xt.values, return_theta=True)[2]
    dg = {}
    env, imps, heavy = smc_envelope(T, E, Z, Xt, miss, rep, theta0=th0, diag=dg)
    e0, l0, h0 = env.eval(np.zeros(env.q))
    out = {"MI, SMC-FCS": (e0, l0, h0)}
    (l, h), _ = env.vertex_2q3(DMAX); out["HPPSM v2 (2q+3 configurations)"] = (e0, l, h)
    (la, ha), _ = env.adaptive(DMAX); out["HPPSM v2 (adaptive)"] = (e0, la, ha)
    (lv, hv), _ = env.vertices(DMAX); out["HPPSM v2 (vertices only)"] = (e0, lv, hv)
    out["HPPSM v2 (proposed)"] = (e0, np.minimum(lv, la), np.maximum(hv, ha))           # vertices + axes
    if scen not in MECH:
        for dm in (0.25, 1.0):
            (l, h), _ = env.adaptive(dm); out[f"HPPSM v2 (dmax = {dm})"] = (e0, l, h)
    rows = pd.DataFrame([dict(rep=rep, method=k, term=t, est=e[i], lo=l_[i], hi=h_[i], truth=truth[t])
                         for k, (e, l_, h_) in out.items() for i, t in enumerate(TERMS)])
    cols = list(LABS.index); Xm = Xt.mask(miss)
    dl = {v: float(np.mean([np.mean(Xt[v].values[miss[v].values] - A[miss[v].values, cols.index(v)]) for A in imps]) / Xm[v].std()) for v in heavy}
    info = dict(rep=rep, n_configs=len(env.cache), **{f"smc_{k}": v for k, v in dg.items()}, **{f"delta_{v}": dl[v] for v in heavy})
    return rows, info


def cindex_rep(tr, rep, mech, n):
    """C-index of each method on an independent test sample of 3,000 patients with true covariates."""
    from sksurv.metrics import concordance_index_censored
    from lifelines import NelsonAalenFitter
    gamma, ydep = MECH[mech]
    rng, rows, T, E, Xt, Xref, miss, Z = generate(tr, 900_000 + 1000 * rep + n + {"M1": 1, "M2": 2, "M3": 3, "M4": 4}[mech], n, gamma, ydep)
    trow = rng.choice(len(tr["Z"]), 3000, replace=False)
    Tt, Et, _, _ = simulate_times(tr, trow, rng)
    Zt = tr["Z"].iloc[trow].reset_index(drop=True)[Z.columns]; Xtt = tr["Xtrue"].iloc[trow].values
    C = lambda lp: concordance_index_censored(Et.astype(bool), Tt, lp)[0]
    lp = lambda TT, EE, ZZ, XX, Zte=Zt: RoystonParmarPH().fit(TT, EE, ZZ.values, XX).lp(Zte.values, Xtt)
    res = {"Oracle": C(lp(T, E, Z, Xt.values)), "Fill-in": C(lp(T, E, Z, Xref.values))}
    cc = ~miss.any(axis=1).values
    if cc.sum() > 50:
        Zc = Z[cc].loc[:, Z[cc].std() > 0]
        try:
            res["Complete case"] = C(lp(T[cc], E[cc], Zc, Xt.values[cc], Zt[Zc.columns]))
        except Exception:
            pass
    na = NelsonAalenFitter().fit(T, E).cumulative_hazard_at_times(T).values; Xm = Xt.mask(miss)
    res["MI, chained equations"] = C(np.mean([lp(T, E, Z, IterativeImputer(random_state=k, sample_posterior=True, max_iter=5).fit_transform(
        np.hstack([Z.values, E[:, None], na[:, None], Xm.values]))[:, Z.shape[1] + 2:]) for k in range(3)], 0))
    res["HPPSM v2 (proposed)"] = C(np.mean([lp(T, E, Z, A) for A in smcfcs_rp(T, E, Z.values, Xm.values, LO_SUP, HI_SUP, m=3, seed=rep, n_cycles=6)], 0))
    return pd.DataFrame([dict(mech=mech, n=n, rep=rep, method=k, C=v) for k, v in res.items()])


def dense_rep(tr, mech, rep, n_random=100):
    gamma, ydep = MECH[mech]
    _, _, T, E, Xt, Xref, miss, Z = generate(tr, 10_000 + rep, 3000, gamma, ydep)
    th0 = rp_terms(T, E, Z, Xt.values, return_theta=True)[2]
    env, imps, heavy = smc_envelope(T, E, Z, Xt, miss, rep, theta0=th0)
    t0 = time.time(); (l1, h1), n1 = env.vertex_2q3(DMAX)
    (l2, h2), n2 = env.adaptive(DMAX)
    (l4, h4), n4 = env.vertices(DMAX)
    l5, h5 = np.minimum(l4, l2), np.maximum(h4, h2)          # vertices + axes (reference, singles, adaptive vertices)
    (l3, h3), n3 = env.dense(DMAX, n_random, np.random.default_rng(rep))
    # union of everything evaluated (dense + adaptive) is the best available approximation of the full union
    keys = list(env.cache); L = np.min([env.cache[k][1] for k in keys], 0); H = np.max([env.cache[k][2] for k in keys], 0)
    out = []
    for i, t in enumerate(TERMS):
        out.append(dict(mech=mech, rep=rep, term=t, truth=tr["true_coef"][t], lo_full=L[i], hi_full=H[i],
                        lo_2q3=l1[i], hi_2q3=h1[i], lo_adapt=l2[i], hi_adapt=h2[i], lo_dense=l3[i], hi_dense=h3[i], lo_vert=l4[i], hi_vert=h4[i], lo_va=l5[i], hi_va=h5[i],
                        n_2q3=n1, n_adapt=n2, n_all=len(keys)))
    return pd.DataFrame(out)


def pseudo_truth(tr, quad, K=20):
    """Pseudo-true coefficients of the (misspecified) linear working model: average of oracle fits to K
    simulated full cohorts with true covariates."""
    est = []
    for k in range(K):
        rng = np.random.default_rng(424242 + k); rows = np.arange(len(tr["Z"]))
        T, E, _, _ = simulate_times(tr, rows, rng, quad)
        Z = tr["Z"].loc[:, tr["Z"].std() > 0]
        est.append(rp_terms(T, E, Z, tr["Xtrue"].values)[0])
    return dict(zip(TERMS, np.mean(est, 0)))

In [ ]:
TR = build_truth()
print("True coefficients:", {k: round(v, 4) for k, v in TR["true_coef"].items()})

import time
def run_resumable(path, reps, fn, key="rep", start=0):
    """Append one replicate at a time to `path`; replicates already present are skipped."""
    done = set(pd.read_csv(path)[key]) if os.path.exists(path) else set()
    for r in range(start, reps):
        if r in done:
            continue
        t0 = time.time(); res = fn(r)
        rows, info = res if isinstance(res, tuple) else (res, None)
        rows.to_csv(path, mode="a", header=not os.path.exists(path), index=False)
        if info is not None:
            ip = path.replace(".csv", "_info.csv"); pd.DataFrame([info]).to_csv(ip, mode="a", header=not os.path.exists(ip), index=False)
        print(f"{os.path.basename(path)}: replicate {r} ({time.time()-t0:.0f} s)")

def summary_table(d, order=None, oracle="Oracle"):
    s, g = coverage_summary(d.drop_duplicates(["rep", "method", "term"]), oracle)
    s["mcse_of_mean"] = g.groupby("method").mcse.mean()
    return s.reindex(order) if order else s

## 4. Main simulation (Table 2, Fig. 2 left)

In [ ]:
NON_SMC = ["Oracle", "Fill-in Wald", "Complete case", "MI, chained equations"]

def load_s2(scen):
    """Non-SMC methods from run_rep; every SMC-FCS-based method (exact truncated proposals, explicit fall-back)
    from smc_rep. The primary HPPSM interval ('HPPSM v2 (proposed)') is computed by vertices + axes."""
    d = pd.read_csv(f"{OUTDIR}/{scen}_0.csv").drop_duplicates(["rep", "method", "term"]); d = d[d.method.isin(NON_SMC)]
    v = pd.read_csv(f"{OUTDIR}/{scen}_smc.csv").drop_duplicates(["rep", "method", "term"])
    return pd.concat([d, v[v.rep.isin(d.rep)]])

for m in MECH:
    run_resumable(f"{OUTDIR}/{m}_0.csv", MAIN_REPS, lambda r, m=m: run_rep(TR, m, r))
    run_resumable(f"{OUTDIR}/{m}_smc.csv", MAIN_REPS, lambda r, m=m: smc_rep(TR, m, r))
info = pd.concat([pd.read_csv(f"{OUTDIR}/{m}_smc_info.csv") for m in MECH])
print(f"SMC-FCS draws not accepted within 1,000 proposals: {info.smc_not_accepted.sum()} of {info.smc_draws.sum()} "
      f"({100 * info.smc_not_accepted.sum() / info.smc_draws.sum():.3f}%); proposals per draw {info.smc_proposals.sum() / info.smc_draws.sum():.1f}")
ORDER = ["Oracle", "Fill-in Wald", "Complete case", "MI, chained equations", "MI, SMC-FCS", "HPPSM v2 (2q+3 configurations)", "HPPSM v2 (adaptive)", "HPPSM v2 (proposed)"]
tab = {m: summary_table(load_s2(m), ORDER).apply(
    lambda r: f"{100*r.mean_coverage:.1f}% ({100*r.lowest_coverage:.0f}%) · {r.width_vs_oracle:.2f}×" if pd.notna(r.mean_coverage) else "–", axis=1) for m in MECH}
table2 = pd.DataFrame(tab); table2.to_csv(f"{OUTDIR}/table2.csv"); table2

In [ ]:
import matplotlib.pyplot as plt
COL = {"Fill-in Wald": "#C0392B", "Complete case": "#8E44AD", "MI, chained equations": "#7F8C8D", "MI, SMC-FCS": "#2E86C1",
       "HPPSM v2 (2q+3 configurations)": "#F5B041", "HPPSM v2 (adaptive)": "#D35400", "HPPSM v2 (proposed)": "#E67E22"}
fig, ax = plt.subplots(figsize=(5.5, 7.5)); y = 0; ticks, labels = [], []
for m in list(MECH)[::-1]:
    _, g = coverage_summary(load_s2(m))
    st = g.groupby("method").cvg.agg(["mean", "min", "max"])
    for k in [k for k in COL if k in st.index][::-1]:
        r = st.loc[k]; ax.plot([r["min"], r["max"]], [y, y], color=COL[k], lw=1.6)
        ax.plot(r["mean"], y, "D" if "proposed" in k else "o", color=COL[k], ms=5, mec="white"); ticks.append(y); labels.append(k); y += 1
    ax.text(0, y + 0.05, m, transform=ax.get_yaxis_transform(), fontweight="bold"); y += 1.6
band = 1.96 * (0.95 * 0.05 / max(MAIN_REPS, 1)) ** 0.5
ax.axvspan(0.95 - band, 0.95 + band, color="#D5E8D4", alpha=0.6); ax.axvline(0.95, ls="--", color="#2E7D32")
ax.set_yticks(ticks); ax.set_yticklabels(labels, fontsize=8); ax.set_xlim(-0.02, 1.02); ax.set_xlabel("Coverage of nominal 95% intervals")
plt.savefig(f"{OUTDIR}/fig2_left.png", dpi=300, bbox_inches="tight"); plt.show()

## 5. Extended scenarios (Table 3, Fig. 3)
Smaller samples, missingness ×0.5 and ×1.5, self-masking MNAR (λ = 0.5, 1, 2) and a misspecified outcome model (pseudo-true estimand), with δ<sub>max</sub> = 0.25, 0.5 and 1.0. The realised MNAR shift is saved in the `_info.csv` files.

In [ ]:
for s in EXT_SCENARIOS:
    truth = None
    if EXT[s][5]:
        pt = f"{OUTDIR}/pseudo_truth_{s}.json"
        if not os.path.exists(pt):
            json.dump(pseudo_truth(TR, EXT[s][5]), open(pt, "w"))
        truth = json.load(open(pt))
    run_resumable(f"{OUTDIR}/{s}_0.csv", EXT_REPS, lambda r, s=s, truth=truth: run_rep(TR, s, r, truth))
    run_resumable(f"{OUTDIR}/{s}_smc.csv", EXT_REPS, lambda r, s=s, truth=truth: smc_rep(TR, s, r, truth))
EM = ["Oracle", "Complete case", "MI, SMC-FCS", "HPPSM v2 (dmax = 0.25)", "HPPSM v2 (proposed)", "HPPSM v2 (dmax = 1.0)"]
rows = {}
for s in EXT_SCENARIOS:
    sm = summary_table(load_s2(s), EM)
    inf = pd.read_csv(f"{OUTDIR}/{s}_smc_info.csv"); shift = inf[[c for c in inf if c.startswith("delta_")]].mean().mean()
    rows[s] = {**{k: f"{100*r.mean_coverage:.0f}% ({100*r.lowest_coverage:.0f}%)" for k, r in sm.iterrows() if pd.notna(r.mean_coverage)}, "realised shift (SD)": round(shift, 2)}
table3 = pd.DataFrame(rows).T; table3.to_csv(f"{OUTDIR}/table3.csv"); table3

## 6. Finite configurations against a dense search (Table 4)
For each replicate: the 2*q* + 3 configurations, the adaptive vertex search, vertices + axes (primary), and all 2<sup>7</sup> vertices plus 100 random points. The shortfall is measured relative to the hull of everything evaluated.

In [ ]:
for m in MECH:
    run_resumable(f"{OUTDIR}/dense_{m}.csv", DENSE_REPS, lambda r, m=m: dense_rep(TR, m, r))
res = []
for m in MECH:
    d = pd.read_csv(f"{OUTDIR}/dense_{m}.csv"); w = d.hi_full - d.lo_full
    for tag in ("2q3", "adapt", "va"):
        sh = (np.maximum(d[f"lo_{tag}"] - d.lo_full, 0) + np.maximum(d.hi_full - d[f"hi_{tag}"], 0)) / w
        res.append(dict(mechanism=m, search=tag, reps=d.rep.nunique(), mean_shortfall_pct=100 * sh.mean(), max_shortfall_pct=100 * sh.max(),
                        share_over_half_pct=100 * (sh > 0.005).mean(), coverage=((d[f"lo_{tag}"] <= d.truth) & (d.truth <= d[f"hi_{tag}"])).mean()))
table4 = pd.DataFrame(res).round(3); table4.to_csv(f"{OUTDIR}/table4.csv"); table4

## 7. Discrimination (Fig. 7, SUPPORT2 panels)
Coefficients from each method are applied to an independent test sample of 3,000 patients with **true** covariates (Harrell's C).

In [ ]:
path = f"{OUTDIR}/cindex_support2.csv"
done = set(map(tuple, pd.read_csv(path)[["mech", "n", "rep"]].values)) if os.path.exists(path) else set()
for rep in range(CIDX_REPS):
    for mech in MECH:
        for n in (500, 1000, 3000):
            if (mech, n, rep) not in done:
                cindex_rep(TR, rep, mech, n).to_csv(path, mode="a", header=not os.path.exists(path), index=False)
pd.read_csv(path).groupby(["mech", "method", "n"]).C.agg(["mean", "count"]).round(3)

## Session information

In [ ]:
import sys, scipy, sklearn, lifelines, sksurv
print(sys.version); print({m.__name__: m.__version__ for m in (np, pd, scipy, sklearn, lifelines, sksurv)})